In [1]:
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
import os

#print the top 3 choerence scores with their respective topics
def output_final(vectorizer, topic_model):
    texts, custom_vect, matrix = text_vectorizer(vectorizer, ngram_range, words_to_remove, documents, max_features, min_frequency)
    if topic_model == LatentDirichletAllocation :
        topic_ranges = lda_range
    elif topic_model == TruncatedSVD:
        topic_ranges = lsa_range
        
    suggest_num_topic(topic_ranges, matrix, custom_vect, texts, top_coherence_score, words_per_topic, topic_model)

# perform text vectorization with the choosen algorithm and the custom analyzer    
def text_vectorizer(vectorizer, ngrams, words_to_remove, documents, features, min_freq):
    base_analyzer = vectorizer(ngram_range=ngrams).build_analyzer()
    customized_analyzer = custom_analyzer(base_analyzer, words_to_remove)
    texts = [customized_analyzer(doc) for doc in documents]
    vect = vectorizer(analyzer=customized_analyzer, max_features=features, min_df=min_freq)
    sparse_matrix = vect.fit_transform(documents)
    return texts, vect, sparse_matrix
    
#Take the single words and 2-grams and remove the one contained in to_remove and returns the rest
def custom_analyzer(embedder_analyzer, words_to_remove):
    def analyzer(doc):
        return [t for t in embedder_analyzer(doc) if t not in words_to_remove]
    return analyzer

#get the number of CPU cores available to the program
def get_usable_cores():
    available_cores = os.cpu_count()
    if available_cores > 1:
        usable_cores = available_cores -1
    else:
        usable_cores = 1
    return usable_cores


def lda_extract_topic(num_topics, cores, doc_term_matrix):
    topic_modeling = LatentDirichletAllocation(n_components=num_topics, random_state=21, n_jobs = cores)
    return topic_modeling.fit(doc_term_matrix)

def lsa_extract_topic(k, doc_term_matrix):
    topic_modeling = TruncatedSVD(n_components=k, random_state=21)
    return topic_modeling.fit(doc_term_matrix)    

    
def suggest_num_topic(topic_range, matrix, vectorizer, text, top_coherence_score, words_per_topic, topic_model):
    results = []
    min_topics, max_topics, step = topic_range
    dictionary = Dictionary(text)
    for topics in range(min_topics, max_topics + 1, step):
        if topic_model == LatentDirichletAllocation:
            model = lda_extract_topic(topics, get_usable_cores(), matrix)
        else:
            model = lsa_extract_topic(topics, matrix)
        score = coherence_score(model, text, dictionary, words_per_topic, vectorizer, get_usable_cores())
        results.append((score, topics, model))
        
    best_results = best_number_topic(results, top_coherence_score, vectorizer, words_per_topic)

#Compute the choerence score for a specific number of topics or dimensions
def coherence_score(model, text, dictionary, top_n, vectorizer, cores):
    feature_names = vectorizer.get_feature_names_out()
    topics = [[str(feature_names[i]) for i in comp.argsort()[::-1][:top_n]] for comp in model.components_]
    window = 10
    cm = CoherenceModel(topics=topics, texts=text, dictionary=dictionary, coherence="c_v", window_size= window, processes = cores )
    score = cm.get_coherence()
    return score

 
def print_topics(choosen_model, vectorizer, top_n):
    words = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(choosen_model.components_):
        print(f"Topic {topic_idx+1}: ", " -".join([words[i] for i in topic.argsort()[:-top_n - 1:-1]]))

#print the topics for the selected scores
def best_number_topic(results,top_best ,vectorizer , top_n):
    best = sorted(results, key=lambda t: t[0], reverse=True)[:top_best]
    for score, n_topics, model in best:
        if isinstance(model, LatentDirichletAllocation):
            print(f"\n==== {n_topics} topics with coherence score using c_v: {score}====")
        elif isinstance(model, TruncatedSVD):
            print(f"\n==== {n_topics} latent dimensions with coherence score using c_v: {score}====")
            
        print_topics(model, vectorizer, top_n)
    return [(score, n_topics) for score, n_topics, _ in best]



    
input_file = "cleaned_review.txt"
with open(input_file, "r", encoding="utf-8") as file:

    documents = [line.strip() for line in file if line.strip()]

words_to_remove = {"no", "not", "nor"}
ngram_range = (1, 2)
max_features = 5000
min_frequency = 100
top_coherence_score = 3
words_per_topic = 10
lda_range = (2, 36, 2)   # (min, max, step)
lsa_range = (2, 36, 2)


In [2]:
tfidf_vectorizer = TfidfVectorizer
lda_topic_model = LatentDirichletAllocation
output_final(tfidf_vectorizer, lda_topic_model)


==== 16 topics with coherence score using c_v: 0.6400344061949867====
Topic 1:  charge -charger -cable -device -plug -work -car -port -cord -use
Topic 2:  case -button -fit -phone -cover -like -back -hard -side -feel
Topic 3:  love -case -color -recommend -buy -great -one -cute -get -would
Topic 4:  phone -use -ape -get -work -go -android -screen -good -one
Topic 5:  like -look -cheap -get -cover -case -good -picture -not like -really
Topic 6:  get -one -break -buy -work -money -month -last -week -would
Topic 7:  stylus -pen -tip -use -ipad -work -tablet -screen -touch -finger
Topic 8:  mount -phone -hold -holder -arm -suction -car -use -windshield -dash
Topic 9:  phone -clip -belt -dock -case -use -stand -work -hold -belt clip
Topic 10:  screen -bubble -protector -install -easy -get -put -easy install -one -screen protector
Topic 11:  screen -protector -screen protector -apply -bubble -easy -scratch -get -glare -one
Topic 12:  work -price -great -product -good -quality -buy -work gre

In [3]:
bow_vectorizer = CountVectorizer
lsa_topic_model = TruncatedSVD
output_final(bow_vectorizer, lsa_topic_model)


==== 2 latent dimensions with coherence score using c_v: 0.5624114338575867====
Topic 1:  phone -case -use -charge -one -battery -get -like -good -screen
Topic 2:  case -phone -screen -protector -screen protector -protection -fit -cover -look -protect

==== 8 latent dimensions with coherence score using c_v: 0.5019898411411434====
Topic 1:  phone -case -use -charge -one -battery -get -like -good -screen
Topic 2:  case -phone -screen -protector -screen protector -protection -fit -cover -look -protect
Topic 3:  case -charge -battery -charger -power -port -iphone -cable -device -fit
Topic 4:  phone -battery -charge -case -charge phone -ape -phone not -phone case -charger -android
Topic 5:  screen -protector -screen protector -charge -battery -phone -bubble -charger -apply -dust
Topic 6:  battery -day -get -life -hour -man -battery life -good -headset -last
Topic 7:  one -get -charger -work -like -buy -would -go -look -great
Topic 8:  use -iphone -device -ape -stylus -android -camera -wor